In [1]:
# =========================================== #
# 03_preprocessing.ipynb                      #
# Preprocessing pipeline for Phase 1 datasets #
# Superset, DETOXIS, HAHA                     #
# =========================================== #

import re
import pandas as pd
import numpy as np
from google.cloud import storage
import spacy
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', '..', 'src')))

# Reproducibility
from utils import set_seed
SEED = 42
set_seed(SEED)

# SpaCy Spanish model for stopword removal 
# only to be used for TF-IDF version (for ML models)
nlp_es = spacy.load('es_core_news_sm')
spanish_stopwords = nlp_es.Defaults.stop_words
extra_stopwords = {'link', 'rt', 'user', 'http', 'https', 'amp', 'via'}
all_stopwords = spanish_stopwords | extra_stopwords

print("Imports OK")
print(f"SpaCy model loaded: {nlp_es.meta['name']}")

Imports OK
SpaCy model loaded: core_news_sm


In [2]:
# ==================== #
# Find datasets in GCP #
# ==================== #

GCS_BUCKET = 'sinodio-raw-data'

client = storage.Client()
bucket = client.bucket(GCS_BUCKET)

blobs = client.list_blobs(GCS_BUCKET)
for blob in blobs:
    print(blob.name)

detoxis/train.csv
haha/train.csv
spanish_hate_superset/train.csv


In [3]:
# ====================== #
# Load datasets from GCP #
# ====================== #

GCS_BUCKET = 'sinodio-raw-data'

client = storage.Client()
bucket = client.bucket(GCS_BUCKET)

def load_csv_from_gcs(bucket, blob_name):
    blob = bucket.blob(blob_name)
    data = blob.download_as_bytes()
    return pd.read_csv(pd.io.common.BytesIO(data))

superset = load_csv_from_gcs(bucket, 'spanish_hate_superset/train.csv')
detoxis  = load_csv_from_gcs(bucket, 'detoxis/train.csv')
haha     = load_csv_from_gcs(bucket, 'haha/train.csv')

print(f"Superset: {superset.shape}")
print(f"DETOXIS:  {detoxis.shape}")
print(f"HAHA:     {haha.shape}")

Superset: (29855, 7)
DETOXIS:  (3463, 3)
HAHA:     (24000, 3)


In [4]:
# ============================== #
# Standardize columns and labels #
# ============================== #

# Superset: rename 'labels' to 'label' and convert float to int
superset = superset[['text', 'labels']].rename(columns={'labels': 'label'})
superset['label'] = superset['label'].astype(int)
superset['source'] = 'superset'

# DETOXIS: already has 'text' and 'label', keep as is
detoxis = detoxis[['text', 'label']]
detoxis['source'] = 'detoxis'

# HAHA: already has 'text' and 'label', keep as is
haha = haha[['text', 'label']]
haha['source'] = 'haha'

print("Superset columns:", superset.columns.tolist())
print("DETOXIS columns: ", detoxis.columns.tolist())
print("HAHA columns:    ", haha.columns.tolist())

print("\nSuperset label dtype:", superset['label'].dtype)
print("DETOXIS label dtype: ", detoxis['label'].dtype)
print("HAHA label dtype:    ", haha['label'].dtype)

print("\nSuperset label values:", sorted(superset['label'].unique()))
print("DETOXIS label values: ", sorted(detoxis['label'].unique()))
print("HAHA label values:    ", sorted(haha['label'].unique()))

Superset columns: ['text', 'label', 'source']
DETOXIS columns:  ['text', 'label', 'source']
HAHA columns:     ['text', 'label', 'source']

Superset label dtype: int64
DETOXIS label dtype:  int64
HAHA label dtype:     int64

Superset label values: [np.int64(0), np.int64(1)]
DETOXIS label values:  [np.int64(0), np.int64(1)]
HAHA label values:     [np.int64(0), np.int64(1)]


In [5]:
# ======================================== #
# Handle duplicates and conflicting labels #
# ======================================== #

print("\n=== BEFORE CLEANING ===")
print(f"Superset: {superset.shape}")
print(f"DETOXIS:  {detoxis.shape}")
print(f"HAHA:     {haha.shape}")

# --- Exact duplicates ---
superset = superset.drop_duplicates(subset=['text', 'label'])
detoxis  = detoxis.drop_duplicates(subset=['text', 'label'])
haha     = haha.drop_duplicates(subset=['text', 'label'])

# --- Conflicting labels (same text, different label) ---
# Keep only texts that have a single unique label
def remove_conflicting_labels(df):
    label_counts = df.groupby('text')['label'].nunique()
    conflicting = label_counts[label_counts > 1].index
    return df[~df['text'].isin(conflicting)]

superset = remove_conflicting_labels(superset)
detoxis  = remove_conflicting_labels(detoxis)
haha     = remove_conflicting_labels(haha)

# --- Duplicates across (DETOXIS and HAHA) ---
detoxis_texts = set(detoxis['text'])
haha_texts    = set(haha['text'])
overlap       = detoxis_texts & haha_texts
print(f"\nDuplicates across datasets (DETOXIS and HAHA): {len(overlap)} texts")
haha = haha[~haha['text'].isin(overlap)]

print("\n=== AFTER CLEANING ===")
print(f"Superset: {superset.shape}")
print(f"DETOXIS:  {detoxis.shape}")
print(f"HAHA:     {haha.shape}")


=== BEFORE CLEANING ===
Superset: (29855, 3)
DETOXIS:  (3463, 3)
HAHA:     (24000, 3)

Duplicates across datasets (DETOXIS and HAHA): 1 texts

=== AFTER CLEANING ===
Superset: (29652, 3)
DETOXIS:  (3444, 3)
HAHA:     (23999, 3)


In [7]:
# ======================= #
# Remove very short texts #
# ======================= #

MIN_CHARS = 3

print("=== BEFORE ===")
print(f"Superset: {superset.shape}")
print(f"DETOXIS:  {detoxis.shape}")
print(f"HAHA:     {haha.shape}")

# Show texts being removed
for name, df in [('Superset', superset), ('DETOXIS', detoxis), ('HAHA', haha)]:
    short = df[df['text'].str.len() < MIN_CHARS]
    if len(short) > 0:
        print(f"\n{name} — texts shorter than {MIN_CHARS} chars:")
        print(short[['text', 'label']])

# Remove short texts
superset = superset[superset['text'].str.len() >= MIN_CHARS]
detoxis  = detoxis[detoxis['text'].str.len() >= MIN_CHARS]
haha     = haha[haha['text'].str.len() >= MIN_CHARS]

print("\n=== AFTER ===")
print(f"Superset: {superset.shape}")
print(f"DETOXIS:  {detoxis.shape}")
print(f"HAHA:     {haha.shape}")

=== BEFORE ===
Superset: (29652, 3)
DETOXIS:  (3444, 3)
HAHA:     (23999, 3)

DETOXIS — texts shorter than 3 chars:
     text  label
977     X      0
978     V      0
1638   xD      0
2123   ok      0
3443   +1      0

HAHA — texts shorter than 3 chars:
      text  label
11921   :(      0
19994    .      0

=== AFTER ===
Superset: (29652, 3)
DETOXIS:  (3439, 3)
HAHA:     (23997, 3)


In [14]:
# ============= #
# Text cleaning #
# ============= #

def clean_text_minimal(text):
    text = re.sub(r'@USER', ' ', text)          # replace @USER with space
    text = re.sub(r'http\S+', ' ', text)        # replace URLs with space
    text = re.sub(r'\brt\b', ' ', text)         # remove retweet marker with space
    text = re.sub(r'\bamp\b', ' ', text)        # remove HTML artifact with space
    text = re.sub(r'\s+', ' ', text)            # normalize whitespace
    return text.strip()
    
def clean_text_tfidf(text):
    """
    Deeper cleaning for TF-IDF + traditional ML.
    Applies minimal cleaning plus lowercasing, 
    punctuation removal, and stopword removal.
    """
    text = clean_text_minimal(text)
    text = text.lower()
    text = re.sub(r'[^\w\s]', ' ', text)        # remove punctuation
    text = re.sub(r'\d+', ' ', text)            # remove digits
    text = re.sub(r'\s+', ' ', text)            # normalize whitespace
    tokens = [t for t in text.split() 
              if t not in all_stopwords 
              and len(t) >= 3]
    return ' '.join(tokens)

# Apply to all three datasets
for df in [superset, detoxis, haha]:
    df['text_clean'] = df['text'].apply(clean_text_minimal)
    df['text_tfidf'] = df['text'].apply(clean_text_tfidf)

# Verify by showing a sample of each
def show_samples(df, name, n=3):
    print(f"\n{'='*60}")
    print(f"Sample from {name}")
    print(f"{'='*60}")
    for _, row in df.head(n).iterrows():
        print(f"ORIGINAL:    {row['text'][:100]}")
        print(f"TEXT_CLEAN:  {row['text_clean'][:100]}")
        print(f"TEXT_TFIDF:  {row['text_tfidf'][:100]}")
        print("-"*60)

show_samples(superset, 'Superset')
show_samples(detoxis, 'DETOXIS')
show_samples(haha, 'HAHA')


Sample from Superset
ORIGINAL:    Eran tan pero tan feministas que invisibilizaban constantemente a las trabajadoras sexuales, haciénd
TEXT_CLEAN:  Eran tan pero tan feministas que invisibilizaban constantemente a las trabajadoras sexuales, haciénd
TEXT_TFIDF:  feministas invisibilizaban constantemente trabajadoras sexuales haciéndole creer mundo incapaces dec
------------------------------------------------------------
ORIGINAL:    @USER @USER @USER Me carga en lo q se convirtió la 2da vuelta a la gobernación...una flaiterío.
TEXT_CLEAN:  Me carga en lo q se convirtió la 2da vuelta a la gobernación...una flaiterío.
TEXT_TFIDF:  carga convirtió vuelta gobernación flaiterío
------------------------------------------------------------
ORIGINAL:    , ¿Sabrán las femiorcas como @USER y todo el flaiterio mapuchento , que si hay una cultura y socieda
TEXT_CLEAN:  , ¿Sabrán las femiorcas como y todo el flaiterio mapuchento , que si hay una cultura y sociedad abso
TEXT_TFIDF:  sabrán femiorca

In [15]:
# =================================== #
# Merge datasets into a single corpus #
# =================================== #

corpus = pd.concat([superset, detoxis, haha], ignore_index=True)

print(f"Total corpus size: {corpus.shape}")
print(f"\nClass distribution:")
print(corpus['label'].value_counts())
print(f"\nClass distribution (%):")
print(corpus['label'].value_counts(normalize=True).round(3) * 100)
print(f"\nSource distribution:")
print(corpus['source'].value_counts())
print(f"\nNull values:")
print(corpus.isnull().sum())

Total corpus size: (57088, 5)

Class distribution:
label
0    39479
1    17609
Name: count, dtype: int64

Class distribution (%):
label
0    69.2
1    30.8
Name: proportion, dtype: float64

Source distribution:
source
superset    29652
haha        23997
detoxis      3439
Name: count, dtype: int64

Null values:
text          0
label         0
source        0
text_clean    0
text_tfidf    0
dtype: int64


In [16]:
# =============================== #
# Train / Validation / Test split #
#      85%           /    15%
# =============================== #

from sklearn.model_selection import train_test_split

# First split: separate test set (15%)
train_val, test = train_test_split(
    corpus,
    test_size=0.15,
    random_state=SEED,
    stratify=corpus['label']
)

# Second split: validation / train (85%/15% of remaining)
train, val = train_test_split(
    train_val,
    test_size=0.15,
    random_state=SEED,
    stratify=train_val['label']
)

print(f"Train:      {train.shape} — {train['label'].value_counts(normalize=True).round(3).to_dict()}")
print(f"Validation: {val.shape} — {val['label'].value_counts(normalize=True).round(3).to_dict()}")
print(f"Test:       {test.shape} — {test['label'].value_counts(normalize=True).round(3).to_dict()}")

# Save splits locally as parquet files 
# (preserves data types and integrates
#  natively with GCP, BigQuery and Spark)
import os
output_dir = '../../data/processed'
os.makedirs(output_dir, exist_ok=True)

train.to_parquet(f'{output_dir}/train.parquet', index=False)
val.to_parquet(f'{output_dir}/val.parquet', index=False)
test.to_parquet(f'{output_dir}/test.parquet', index=False)

print("\nSplits saved locally to data/processed/")

Train:      (41245, 5) — {0: 0.692, 1: 0.308}
Validation: (7279, 5) — {0: 0.692, 1: 0.308}
Test:       (8564, 5) — {0: 0.691, 1: 0.309}

Splits saved locally to data/processed/


The class distribution looks consistent across all three datasets (thanks to stratifying by label), and this should be manageable by the ML models using class_weight = "balanced", which adjusts the loss function during training so the model doesn't simply learn to predict the majority class.

In [17]:
# ============================== #
# Upload processed splits to GCS #
# sinodio-processed-data bucket  #
# ============================== #

GCS_PROCESSED_BUCKET = 'sinodio-processed-data'
processed_bucket = client.bucket(GCS_PROCESSED_BUCKET)

splits = {
    'train': train,
    'val':   val,
    'test':  test
}

for split_name, df in splits.items():
    local_path = f'{output_dir}/{split_name}.parquet'
    blob_path  = f'phase1/{split_name}.parquet'
    blob = processed_bucket.blob(blob_path)
    blob.upload_from_filename(local_path)
    print(f"Uploaded {split_name}.parquet → gs://{GCS_PROCESSED_BUCKET}/{blob_path}")

print("\nAll splits uploaded to GCS.")

Uploaded train.parquet → gs://sinodio-processed-data/phase1/train.parquet
Uploaded val.parquet → gs://sinodio-processed-data/phase1/val.parquet
Uploaded test.parquet → gs://sinodio-processed-data/phase1/test.parquet

All splits uploaded to GCS.


**Comment about upload to GCS:**

All three parquet files are correctly uploaded to Google Cloud Storage.

In [21]:
# ======================================== #
# Spark ETL: compute preprocessing metrics #
# and write summary to BigQuery            #
# ======================================== #

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from google.cloud import bigquery
import datetime

# --- Start Spark session ---
spark = SparkSession.builder \
    .appName("sinodio-preprocessing-metrics") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

# --- Load parquet splits into Spark ---
split_paths = {
    'train': f'{output_dir}/train.parquet',
    'val':   f'{output_dir}/val.parquet',
    'test':  f'{output_dir}/test.parquet'
}

spark_splits = {
    name: spark.read.parquet(path)
    for name, path in split_paths.items()
}

# --- Compute metrics per split ---
rows = []

for split_name, sdf in spark_splits.items():
    total      = sdf.count()
    n_hate     = sdf.filter(F.col('label') == 1).count()
    n_not_hate = sdf.filter(F.col('label') == 0).count()
    pct_hate   = round(n_hate / total * 100, 2)
    avg_len    = sdf.select(F.avg(F.length('text_clean'))).collect()[0][0]
    
    source_dist = (
        sdf.groupBy('source')
           .count()
           .orderBy(F.desc('count'))
           .toPandas()
           .set_index('source')['count']
           .to_json()
    )

    rows.append({
        'split':        split_name,
        'n_rows':       total,
        'n_hate':       n_hate,
        'n_not_hate':   n_not_hate,
        'pct_hate':     pct_hate,
        'avg_text_len': round(avg_len, 1),
        'source_dist':  source_dist,
        'created_at':   datetime.datetime.now(datetime.timezone.utc).isoformat()
    })
    
    print(f"{split_name:6s} → {total:6d} rows | hate: {pct_hate}% | avg_len: {round(avg_len,1)} chars")

spark.stop()

# --- Write to BigQuery (via native client after Spark computes) ---
bq_client = bigquery.Client()
table_id = 'project-5c89dcac-34cb-453d-bd7.sinodio_results.preprocessing_summary'

# Create table if it doesn't exist
schema = [
    bigquery.SchemaField('split',         'STRING'),
    bigquery.SchemaField('n_rows',        'INTEGER'),
    bigquery.SchemaField('n_hate',        'INTEGER'),
    bigquery.SchemaField('n_not_hate',    'INTEGER'),
    bigquery.SchemaField('pct_hate',      'FLOAT'),
    bigquery.SchemaField('avg_text_len',  'FLOAT'),
    bigquery.SchemaField('source_dist',   'STRING'),
    bigquery.SchemaField('created_at',    'STRING'),
]

table = bigquery.Table(table_id, schema=schema)
table = bq_client.create_table(table, exists_ok=True)

errors = bq_client.insert_rows_json(table_id, rows)

if not errors:
    print(f"\nSummary written to BigQuery: {table_id}")
else:
    print(f"BigQuery insert errors: {errors}")

train  →  41245 rows | hate: 30.84% | avg_len: 109.9 chars
val    →   7279 rows | hate: 30.84% | avg_len: 110.5 chars
test   →   8564 rows | hate: 30.85% | avg_len: 110.8 chars

Summary written to BigQuery: project-5c89dcac-34cb-453d-bd7.sinodio_results.preprocessing_summary


**Comment about Spark ETL:**

Again, we see that the metrics look good, with hate class balanced (~30.84%) across all three splits. This confirms that the stratification worked as expected.

The preprocessing summary is correctly uploaded to BigQuery as the table `preprocessing_summary` in the `sinodio_results` dataset.